In [17]:
import re
import os
import sys
import pandas as pd
from datetime import *

In [18]:
# BEAST2 unfortunately does not handle uncertain tip dates so well
# To best sample from a range of dates for tip date uncertainty, you need to add three blocks to your XML manually

In [19]:
# # Function: read in a list of strains with ambiguous dates
# def read_strains(STRAINLISTPATH):
#     OBJECTS = open(STRAINLISTPATH, "r")

#     obj_to_include = OBJECTS.read()

#     OBJECTS.close()

#     strain_list = obj_to_include.split("\n")

#     return(strain_list)

In [20]:
# Function: read in a list of strains with ambiguous dates

def read_strains(STRAINLISTPATH):
    
    strain_list_df = pd.read_csv(STRAINLISTPATH, sep = "\t", header = None)

    strain_list = list(strain_list_df[0])

    return(strain_list)

In [21]:
month_dict = {"01":31,
              "02":28,
              "03":31,
              "04":30,
              "05":31,
              "06":30,
              "07":31,
              "08":31,
              "09":30,
              "10":31,
              "11":30,
              "12":31
             }

In [22]:
# Function to convert date to decimal year
def to_decimal_year(date):
    year = date.year
    start_of_year = datetime(year, 1, 1)
    end_of_year = datetime(year + 1, 1, 1)
    year_length = (end_of_year - start_of_year).days
    day_of_year = (date - start_of_year).days
    return year + day_of_year / year_length

In [23]:
# Function: determine the upper and lower bounds for each date 
def determine_date_bounds(date):
    date = str(date)
    year = date.split("-")[0]
    month = date.split("-")[1]
    
    date_lower = year + "-" + month + "-" + "01"
    n_days = month_dict[month]
    if n_days == 28:
        date_upper = year + "-" + month + "-" + "28"
    elif n_days == 30:
        date_upper = year + "-" + month + "-" + "30"
    elif n_days == 31:
        date_upper = year + "-" + month + "-" + "31"
    else:
        print(f"unexpected date: {date}")
        
    # convert to datetime
    datetime_lower = datetime.strptime(date_lower, "%Y-%m-%d")
    datetime_upper = datetime.strptime(date_upper, "%Y-%m-%d")
    
    # generate decimal date
    decimal_date_lower = to_decimal_year(datetime_lower)
    decimal_date_upper = to_decimal_year(datetime_upper)

    return(decimal_date_lower, decimal_date_upper)

In [24]:
def generate_distribution_block(STRAIN, TREE, COUNT):

    line1 = "            <distribution id=\"insert_prior_id\" spec=\"beast.base.evolution.tree.MRCAPrior\" tipsonly=\"true\" tree=\"@Tree.t:insert_tree_id\">"
    line2 = "                <taxonset id=\"insert_id\" spec=\"TaxonSet\">"
    line3 = "                    <taxon id=\"insert_taxon_id\" spec=\"Taxon\"/>"
    line4 = "                </taxonset>"
    line5 = "                <Uniform id=\"Uniform.insert_count_here\" lower=\"lower_prior_bound_year\" name=\"distr\" upper=\"upper_prior_bound_year\"/>"
    line6 = "            </distribution>"

    tree_id = str(TREE)
    strain_id = str(STRAIN)
    strain_date = strain_id.split("|")[4]
    decimal_date_lower, decimal_date_upper = determine_date_bounds(strain_date)
    taxonset_id = strain_id.split("|")[0]
    taxonset_id_prior = taxonset_id + ".prior"
    count = COUNT
    
    line1 = line1.replace("insert_prior_id",taxonset_id_prior)
    line1 = line1.replace("insert_tree_id",tree_id)
    line2 = line2.replace("insert_id",taxonset_id)
    line3 = line3.replace("insert_taxon_id",strain_id)
    line5 = line5.replace("insert_count_here", str(count))
    line5 = line5.replace("lower_prior_bound_year",str(decimal_date_lower))
    line5 = line5.replace("upper_prior_bound_year",str(decimal_date_upper))

    block = "\n".join([line1,line2,line3,line4,line5,line6])
    
    return(block)

In [25]:
def generate_operator_block(STRAIN, TREE):

    operator_line = "    <operator id=\"tipDatesSampler.insert_strain_id\" spec=\"TipDatesRandomWalker\" taxonset=\"@insert_strain_id\" tree=\"@Tree.t:insert_tree_id\" weight=\"0.01\" windowSize=\"1.0\"/>"
    
    taxonset_id = str(STRAIN).split("|")[0]
    tree_id = str(TREE)
    
    operator_line = operator_line.replace("insert_strain_id",taxonset_id)
    operator_line = operator_line.replace("insert_tree_id",tree_id)

    return(operator_line)

In [26]:
def generate_logger_block(STRAIN):
    
    taxonset_id = str(STRAIN).split("|")[0]
    
    logger_line = "        <log idref=\"insert_strain_name.prior\"/>".replace("insert_strain_name",taxonset_id)
    
    return(logger_line)

In [27]:
def make_blocks(strain_list, TREE, COUNT):

    full_distribution_block = []
    full_operator_block = []
    full_logger_block = []

    for i in strain_list:
        # Generate the appropriate text for each strain
        STRAIN = i
        distribution = generate_distribution_block(STRAIN, TREE, COUNT)
        COUNT += 1
        operator = generate_operator_block(STRAIN, TREE)
        logger = generate_logger_block(STRAIN)

        # Append each line to the correct list
        full_distribution_block.append(distribution)
        full_operator_block.append(operator)
        full_logger_block.append(logger)

    return(full_distribution_block, full_operator_block, full_logger_block)

In [28]:
def write_xml(OUTPUT, INPUT, full_distribution_block, full_operator_block, full_logger_block): 

    with open(OUTPUT, "w") as outfile: 
        outfile.write("")
    
    with open(INPUT, "r") as infile: 
        for line in infile:

            # I do not know what this is for...
            # if "traitname=\"date\"" in line:
            #     with open(output_xml, "a") as outfile: 
            #         outfile.write(new_date_line)

            if "<!--  add tip calibration here -->" in line:
                block_1 = "\n".join(full_distribution_block)
                with open(OUTPUT, "a") as outfile: 
                    outfile.write(line + "\n" + block_1 + "\n" + "<!--  end of tip calibration -->" + "\n\n")
            
            elif "<!-- insert tip calibration operators -->" in line:
                block_2 = "\n".join(full_operator_block)
                with open(OUTPUT, "a") as outfile: 
                    outfile.write(line + block_2 + "\n" + "<!--  end of tip calibration operators -->" + "\n\n")
                
            elif "<!-- insert tip calibration loggers -->" in line:
                block_3 = "\n".join(full_logger_block)
                with open(OUTPUT, "a") as outfile: 
                    outfile.write(line + block_3 + "\n" + "<!--  end of tip calibration loggers-->" + "\n\n")
                
            else: 
                pass
                # with open(output_xml, "a") as outfile: 
                #     outfile.write(line)

In [38]:
# Upload the list of strains with ambiguous dates

strain_list = read_strains("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/inputs/eq-time-host-3000_ambiguous_dates.tsv")
# strain_list = read_strains("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/eq-time-host-1000_ambiguous_dates.tsv")
# strain_list = read_strains("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/eq-time-loc_ambiguous_dates.tsv")
# strain_list = read_strains("../BEAST/empirical_trees/submission_files/2026-01_main-intro_prop-time-host_ha_empirical_targetedbeast/prop-time-host_ambiguous_dates.tsv")
# strain_list = read_strains("../BEAST/empirical_trees/submission_files/2026-01_main-intro_prop-time-loc_ha_empirical_targetedbeast/prop-time-loc_ambiguous_dates.tsv")

In [39]:
COUNT = 5 # Start this count on some number you know will not overlap
TREE = "div-tree-strains-newheader-aligned"

full_distribution_block, full_operator_block, full_logger_block = make_blocks(strain_list, TREE, COUNT)

In [40]:
INPUT = "../BEAST/empirical_trees/submission_files/beast2_tipuncertainty.xml"

OUTPUT = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/inputs/eq-time-host-3000_tipuncertainty.xml"
# OUTPUT = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/eq-time-host-1000_tipuncertainty.xml"
# OUTPUT = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/eq-time-loc_tipuncertainty.xml"
# OUTPUT = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_prop-time-host_ha_empirical_targetedbeast/prop-time-host_tipuncertainty.xml"
# OUTPUT = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_prop-time-loc_ha_empirical_targetedbeast/prop-time-loc_tipuncertainty.xml"

write_xml(OUTPUT, INPUT, full_distribution_block, full_operator_block, full_logger_block)